# Regenie - any ancestry pipeline

Goal is to make a pipeline where you just provide ancestry and it preps everything until running regenie directly!

Regenie seems to be best for maximizing sample size is relatively small cohorts. So let's test:

* AAC
* CAS
* MDE

Now for the true multi-ancestry GWAS, IMO we should do MAF > 0.005 (calculated for controls only).

This is just a test!

---
What is REGENIE?

REGENIE is an analysis program that can be used for Biobank-level GWAS/burden analysis. Why use Regenie? It can account for relatedness to increase sample size, use firth-correction for lopsided case-control ratios, reduce phenotype variance (by accounting for polygenic effect outside of the tested SNP), and you can run multiple phenotypes at the same time efficiently. SAIGE can also account for relatedness but 1. is much slower 2. effect size may be inflated by using SPA correction.

Also unlike SAIGE, the error messages actually make sense.

1. Define phenotypes and participants
2. Prepare regenie step 1 file
3. Run Regenie step 1
4. Run Regenie step 2

Note that step 1 and 2 DO NOT share the same input. Step 1 is used to generate a whole-genome model to account for relatedness, so you will just use unimputed genotyped data (with proper QC). Step 2 can be any data e.g. WGS, imputed, etc. Step 1 and 2 do not have to share file types (plink 1 bed vs plink2 pgen) or even genome build (hg19 vs hg38)!

Some links to help you:

- Regenie documentation: https://rgcgithub.github.io/regenie/
- Regenie manuscript: https://www.nature.com/articles/s41588-021-00870-7
- Github: https://github.com/rgcgithub/regenie/

In [ ]:
# Edit only this section before running the notebook from the top.
from pathlib import Path
import os

ANC = "FIN"
RELEASE = 12 

# Folder in which this notebook should create data/ and output/.
PROJECT_DIR = Path.cwd()

# Root of the GP2 release mounted in verily
GP2_RELEASE_DIR = Path("/home/jupyter/workspace/gp2_tier2_eu_release12")

# Existing hg38 blacklist BED file. This file is read directly, never copied. Jeff gave us this file
BLACKLIST_BED = Path("/home/jupyter/workspace/ws_files/MAMA_2/FirstTryApril26/data/hg38-blacklist.v2.bed")

# Use command names when plink2/regenie are on PATH, or enter full paths.
PLINK2 = "plink2"
REGENIE = "regenie"
THREADS = 8
CHROMOSOMES = range(1, 23)
STEP2_CHROMOSOMES = CHROMOSOMES

# standard settings so the paths will work, not really need to change anything bellow here
PROJECT_DIR = PROJECT_DIR.expanduser().resolve()
GP2_RELEASE_DIR = GP2_RELEASE_DIR.expanduser().resolve()
BLACKLIST_BED = BLACKLIST_BED.expanduser().resolve()
RELEASE_TAG = f"R{RELEASE}"
RELEASE_SUFFIX = f"release{RELEASE}_vwb"
R12_DIR = str(GP2_RELEASE_DIR)  
os.chdir(PROJECT_DIR)
for directory in (PROJECT_DIR / "data", PROJECT_DIR / "output"):
    directory.mkdir(parents=True, exist_ok=True)

if not GP2_RELEASE_DIR.is_dir():
    raise FileNotFoundError(f"GP2_RELEASE_DIR does not exist: {GP2_RELEASE_DIR}")
if not BLACKLIST_BED.is_file():
    raise FileNotFoundError(
        f"BLACKLIST_BED does not exist: {BLACKLIST_BED}. Update it in the config cell."
    )

print(f"Ancestry: {ANC} | Release: {RELEASE} | Threads: {THREADS}")
print(f"Project directory: {PROJECT_DIR}")
print(f"GP2 release directory: {GP2_RELEASE_DIR}")
print(f"Blacklist BED: {BLACKLIST_BED}")


import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from scipy.special import log_ndtr
import sys
import networkx as nx
from typing import Set, List, Tuple, Dict

Ancestry: FIN | Release: 12 | Threads: 8
Project directory: /home/jupyter/workspace/ws_files/MAMA_2/jf_release12/test
GP2 release directory: /home/jupyter/workspace/gp2_tier2_eu_release12
Blacklist BED: /home/jupyter/workspace/ws_files/MAMA_2/FirstTryApril26/data/hg38-blacklist.v2.bed


In [4]:
#os.chdir('')

### Define ancestry

In [5]:
# ANC is set in the collaborator configuration cell above.
print(ANC)

FIN


### Define Release path

In [6]:
# GP2_RELEASE_DIR / R12_DIR are set in the collaborator configuration cell.
print(GP2_RELEASE_DIR)

/home/jupyter/workspace/gp2_tier2_eu_release12


In [7]:
# !mkdir -p ./data/symlink_R12_{ANC}
# !pwd
!pwd
!mkdir -p ./data/symlink_R12_{ANC}

/home/jupyter/workspace/ws_files/MAMA_2/jf_release12/test


In [8]:
print(ANC)

FIN


# 1. Create symlink because psam needs FID

In [9]:
for CHRNUM in CHROMOSOMES:
    CMD = f'{PLINK2} --pfile {R12_DIR}/imputed_genotypes/{ANC}/chr{CHRNUM}_{ANC}_release12_vwb \
    --write-snplist \
    --make-just-psam cols=+fid --out data/symlink_R12_{ANC}/chr{CHRNUM}_{ANC}_release12_vwb'
    !$CMD

PLINK v2.0.0-a.6.9LM 64-bit Intel (29 Jan 2025)    cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to data/symlink_R12_FIN/chr1_FIN_release12_vwb.log.
Options in effect:
  --make-just-psam cols=+fid
  --out data/symlink_R12_FIN/chr1_FIN_release12_vwb
  --pfile /home/jupyter/workspace/gp2_tier2_eu_release12/imputed_genotypes/FIN/chr1_FIN_release12_vwb
  --write-snplist

Start time: Thu Sep  3 14:18:19 2026
52195 MiB RAM detected, ~18434 available; reserving 18370 MiB for main
workspace.
Using up to 8 compute threads.
183 samples (103 females, 80 males; 183 founders) loaded from
/home/jupyter/workspace/gp2_tier2_eu_release12/imputed_genotypes/FIN/chr1_FIN_release12_vwb.psam.
1079262 variants loaded from
/home/jupyter/workspace/gp2_tier2_eu_release12/imputed_genotypes/FIN/chr1_FIN_release12_vwb.pvar.
1 binary phenotype loaded (126 cases, 20 controls).
--write-snplist: Variant IDs written to
data/symlink_R12_FIN/chr1_FIN_re

In [13]:
# CMD1 = f'mkdir -p data/symlink_R12_{ANC}; \
# ln -s $(realpath {R12_DIR}/imputed_genotypes/{ANC}/chr*_{ANC}_release11_vwb.'
# CMD2 = '{pvar,pgen}) '
# CMD3 = f'data/symlink_R12_{ANC}/'
# SYMLINK_CMD = CMD1+CMD2+CMD3
# !bash ls "$SYMLINK_CMD"

SYMLINK_CMD = (
    f"mkdir -p data/symlink_R12_{ANC}; "
    f"ln -s $(realpath {R12_DIR}/imputed_genotypes/{ANC}/chr*_{ANC}_release12_vwb.{{pvar,pgen}}) "
    f"data/symlink_R12_{ANC}/"
)

print(SYMLINK_CMD)
!bash -lc "$SYMLINK_CMD"

mkdir -p data/symlink_R12_FIN; ln -s $(realpath /home/jupyter/workspace/gp2_tier2_eu_release12/imputed_genotypes/FIN/chr*_FIN_release12_vwb.{pvar,pgen}) data/symlink_R12_FIN/
ln: failed to create symbolic link './chr10_FIN_release12_vwb.pvar': File exists
bash: line 2: /home/jupyter/workspace/gp2_tier2_eu_release12/imputed_genotypes/FIN/chr11_FIN_release12_vwb.pvar: Permission denied
bash: line 3: /home/jupyter/workspace/gp2_tier2_eu_release12/imputed_genotypes/FIN/chr12_FIN_release12_vwb.pvar: Permission denied
bash: line 4: /home/jupyter/workspace/gp2_tier2_eu_release12/imputed_genotypes/FIN/chr13_FIN_release12_vwb.pvar: Permission denied
bash: line 5: /home/jupyter/workspace/gp2_tier2_eu_release12/imputed_genotypes/FIN/chr14_FIN_release12_vwb.pvar: Permission denied
bash: line 6: /home/jupyter/workspace/gp2_tier2_eu_release12/imputed_genotypes/FIN/chr15_FIN_release12_vwb.pvar: Permission denied
bash: line 7: /home/jupyter/workspace/gp2_tier2_eu_release12/imputed_genotypes/FIN/chr16_

In [14]:
!ls -lh data/symlink_R12_{ANC}

total 245M
-rw-rw-rw-. 1 jupyter users 1.1K Sep  3 14:18 chr10_FIN_release12_vwb.log
-rw-rw-rw-. 1 jupyter users 6.0K Sep  3 14:18 chr10_FIN_release12_vwb.psam
-rw-rw-rw-. 1 jupyter users  13M Sep  3 14:18 chr10_FIN_release12_vwb.snplist
-rw-rw-rw-. 1 jupyter users 1.1K Sep  3 14:18 chr11_FIN_release12_vwb.log
-rw-rw-rw-. 1 jupyter users 6.0K Sep  3 14:18 chr11_FIN_release12_vwb.psam
-rw-rw-rw-. 1 jupyter users  13M Sep  3 14:18 chr11_FIN_release12_vwb.snplist
-rw-rw-rw-. 1 jupyter users 1.1K Sep  3 14:18 chr12_FIN_release12_vwb.log
-rw-rw-rw-. 1 jupyter users 6.0K Sep  3 14:18 chr12_FIN_release12_vwb.psam
-rw-rw-rw-. 1 jupyter users  13M Sep  3 14:18 chr12_FIN_release12_vwb.snplist
-rw-rw-rw-. 1 jupyter users 1.1K Sep  3 14:18 chr13_FIN_release12_vwb.log
-rw-rw-rw-. 1 jupyter users 6.0K Sep  3 14:18 chr13_FIN_release12_vwb.psam
-rw-rw-rw-. 1 jupyter users 9.3M Sep  3 14:18 chr13_FIN_release12_vwb.snplist
-rw-rw-rw-. 1 jupyter users 1.1K Sep  3 14:18 chr14_FIN_release12_vwb.log
-rw-rw-

In [15]:
print(ANC)

FIN


In [16]:
# CMD1 = f'mkdir -p ./data/symlink_R11_{ANC}; \
# ln -s $(realpath {R11_DIR}/raw_genotypes/{ANC}/{ANC}'
# CMD2 = '_release11_vwb.{pvar,pgen}) '
# CMD3 = f'data/symlink_R11_{ANC}/'
# SYMLINK_CMD = CMD1+CMD2+CMD3
# !bash -lc $SYMLINK_CMD

SYMLINK_CMD = (
    f"mkdir -p data/symlink_R12_{ANC}; "
    f"ln -s $(realpath {R12_DIR}/raw_genotypes/{ANC}/{ANC}_release12_vwb.{{pvar,pgen}}) "
    f"data/symlink_R12_{ANC}/"
)

print(SYMLINK_CMD)
!bash -lc "$SYMLINK_CMD"

mkdir -p data/symlink_R12_FIN; ln -s $(realpath /home/jupyter/workspace/gp2_tier2_eu_release12/raw_genotypes/FIN/FIN_release12_vwb.{pvar,pgen}) data/symlink_R12_FIN/
bash: line 2: /home/jupyter/workspace/gp2_tier2_eu_release12/raw_genotypes/FIN/FIN_release12_vwb.pgen: Permission denied


In [17]:
print(ANC)

FIN


In [18]:
CMD = f'{PLINK2} --pfile {R12_DIR}/raw_genotypes/{ANC}/{ANC}_release12_vwb \
--make-just-psam cols=+fid --out data/symlink_R12_{ANC}/{ANC}_release12_vwb'
!$CMD

PLINK v2.0.0-a.6.9LM 64-bit Intel (29 Jan 2025)    cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to data/symlink_R12_FIN/FIN_release12_vwb.log.
Options in effect:
  --make-just-psam cols=+fid
  --out data/symlink_R12_FIN/FIN_release12_vwb
  --pfile /home/jupyter/workspace/gp2_tier2_eu_release12/raw_genotypes/FIN/FIN_release12_vwb

Start time: Thu Sep  3 14:23:54 2026
52195 MiB RAM detected, ~18424 available; reserving 18360 MiB for main
workspace.
Using up to 8 compute threads.
183 samples (103 females, 80 males; 183 founders) loaded from
/home/jupyter/workspace/gp2_tier2_eu_release12/raw_genotypes/FIN/FIN_release12_vwb.psam.
1 binary phenotype loaded (126 cases, 20 controls).
Writing data/symlink_R12_FIN/FIN_release12_vwb.psam ... done.
End time: Thu Sep  3 14:23:54 2026


# 2. Consult Regenie Manual



In [19]:
print(ANC)

FIN


In [20]:
!{REGENIE} --help

              |=============================|
              |      REGENIE v4.1.2.gz      |
              |=============================|

Copyright (c) 2020-2024 Joelle Mbatchou, Andrey Ziyatdinov and Jonathan Marchini.
Distributed under the MIT License.
Compiled with Boost Iostream library.
Using Intel MKL with Eigen.


Usage:
  regenie [OPTION...]

  -h, --help      print list of available options
      --helpFull  print list of all available options

 Main options:
      --step INT                specify if fitting null model (=1) or 
                                association testing (=2)
      --bed PREFIX              prefix to PLINK .bed/.bim/.fam files
      --pgen PREFIX             prefix to PLINK2 .pgen/.pvar/.psam files
      --bgen FILE               BGEN file
      --sample FILE             sample file corresponding to BGEN file
      --bgi FILE                index bgi file corresponding to BGEN file
      --ref-first               use the first allele as the reference

# 3. Phenotype file

One phenotype file for all ancestries. Don't worry, it shouldn't affect the actual GWAS.

For this test we will include all PD, including enriched, monogenic, etc. However for full GWAS, we should be more selective e.g. subset only to idiopathic case/control group + population control.

In [21]:
print(ANC)

FIN


In [22]:
CLINICAL_DF = pd.read_csv(
    f"{R12_DIR}/clinical_data/master_key_release12_final_vwb.csv"
).rename(columns={'GP2ID':'IID'})
CLINICAL_DF.head()

/tmp/ipykernel_1874/40050047.py:1: DtypeWarning: Columns (31) have mixed types. Specify dtype option on import or set low_memory=False.
  CLINICAL_DF = pd.read_csv(


,study,FID,IID,nba,wgs,clinical_exome,extended_clinical_data,nba_prune_reason,nba_label,wgs_prune_reason,...,family_history_for_qc,region_for_qc,manifest_id,genotyping_site,sample_type,amppd,flag,wgs_ploidy_estimation,variant_report,path_data_available
0,24HR,24HR_000001,24HR_000001,1.0,1.0,NaN,NaN,NaN,EUR,NaN,...,Not Reported,USA,m1,Psomagen,DNA,0.0,NaN,XX,0.0,NaN
1,24HR,24HR_000002,24HR_000002,1.0,1.0,NaN,NaN,NaN,EUR,NaN,...,Not Reported,USA,m1,Psomagen,DNA,0.0,NaN,XY,0.0,NaN
2,24HR,24HR_000003,24HR_000003,1.0,1.0,NaN,NaN,NaN,EUR,NaN,...,Not Reported,USA,m1,Psomagen,DNA,0.0,NaN,XY,0.0,NaN
3,24HR,24HR_000004,24HR_000004,1.0,1.0,NaN,NaN,NaN,EUR,NaN,...,Not Reported,USA,m1,Psomagen,DNA,0.0,NaN,XY,1.0,NaN
4,24HR,24HR_000005,24HR_000005,1.0,1.0,NaN,NaN,NaN,EUR,NaN,...,Not Reported,USA,m1,Psomagen,DNA,0.0,NaN,XY,1.0,NaN


In [23]:
CLINICAL_DF["GP2_phenotype"].value_counts()

GP2_phenotype
PD                       88340
Control                  39900
Population Control       11900
Prodromal                 7878
PSP                       2758
MSA                       1599
Other                     1241
DLB                       1069
AD                         710
CBD/CBS                    405
Mix                        315
Undetermined-MCI           264
LBD                        227
Undetermined-Dementia      140
FTD                        107
VaPD                        26
VaD                         12
Name: count, dtype: int64

In [24]:
CLINICAL_DF["GP2_PHENO"].value_counts()

GP2_PHENO
PD                                79326
Control                           37966
Population Control                11900
Affected_PD                        9014
Prodromal                          7878
PSP                                2720
Unaffected                         1934
MSA                                1580
DLB                                1060
Other                               959
AD                                  705
CBD/CBS                             400
Mix                                 306
Affected_Other                      282
Undetermined-MCI                    264
LBD                                 227
Undetermined-Dementia               139
FTD                                 106
Affected_PSP                         38
VaPD                                 26
Affected_MSA                         19
VaD                                  12
Affected_DLB                          9
Affected_Mix                          9
Affected_CBD/CBS              

In [25]:
CLINICAL_DF_nba_pruned = CLINICAL_DF[
    CLINICAL_DF['nba_prune_reason'].isna()
    & (CLINICAL_DF['nba']==1)
    & (CLINICAL_DF["GP2_PHENO"].isin(['Control','Population Control','PD','Affected_PD'])) # just do PD + Control for case/control, or PD, Control, and Population Control
]
CLINICAL_DF_nba_pruned['PD'] = 0
CLINICAL_DF_nba_pruned.loc[
    CLINICAL_DF_nba_pruned["GP2_PHENO"].isin(['PD','Affected_PD']), # same deal here
    'PD'
] = 1
print(ANC)

FIN


/tmp/ipykernel_1874/2025663080.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  CLINICAL_DF_nba_pruned['PD'] = 0


In [26]:
PLINK_UNIVERSAL_PHENO = CLINICAL_DF_nba_pruned[['FID','IID','PD']]
REGENIE_UNIVERSAL_PHENO = CLINICAL_DF_nba_pruned[['FID','IID','PD']]
PLINK_UNIVERSAL_PHENO['PD'] = PLINK_UNIVERSAL_PHENO['PD']+1
print(ANC)

FIN


/tmp/ipykernel_1874/2177568461.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  PLINK_UNIVERSAL_PHENO['PD'] = PLINK_UNIVERSAL_PHENO['PD']+1


In [27]:
CLINICAL_DF['nba_label'].unique()
print(ANC)

FIN


For plink, remove relateds.

In [28]:

relateds = []
ANCESTRIES = CLINICAL_DF['nba_label'].unique()
ANCESTRIES = ANCESTRIES[~pd.isnull(ANCESTRIES)]
for ancestry in ANCESTRIES:
    related_temp = pd.read_csv(f'{R12_DIR}/meta_data/related_samples/{ancestry}_release12_vwb.related')
    relateds.append(related_temp)
relateds = pd.concat(relateds)


In [29]:
print(ANC)

FIN


In [30]:
PLINK_UNIVERSAL_PHENO[
    ~PLINK_UNIVERSAL_PHENO['IID'].isin(relateds['IID2'])
].shape[0]/PLINK_UNIVERSAL_PHENO[
    ~PLINK_UNIVERSAL_PHENO['IID'].isin(relateds['IID1'])
].shape[0]

0.9970667572104955

In [31]:
PLINK_UNIVERSAL_PHENO_norelated = PLINK_UNIVERSAL_PHENO[
    ~PLINK_UNIVERSAL_PHENO['IID'].isin(relateds['IID1'])
]

In [32]:
! mkdir -p data/COV_PHENO

In [33]:
PLINK_UNIVERSAL_PHENO_norelated.to_csv('data/COV_PHENO/GP2_R12_ALLANC_PD_PHENO.plink.txt', sep = '\t', index=None, na_rep = 'NA')
REGENIE_UNIVERSAL_PHENO.to_csv('data/COV_PHENO/GP2_R12_ALLANC_PD_PHENO.regenie.txt', sep = '\t', index=None, na_rep = 'NA')

# 4. Covariate

Covariates are: SEX, PC1-10, Genotyping sites (optional)

Same deal, single covariate file for all ancestries, but should have no affect on GWAS.

In [34]:
CLINICAL_DF_nba_pruned.tail()

,study,FID,IID,nba,wgs,clinical_exome,extended_clinical_data,nba_prune_reason,nba_label,wgs_prune_reason,...,region_for_qc,manifest_id,genotyping_site,sample_type,amppd,flag,wgs_ploidy_estimation,variant_report,path_data_available,PD
156938,YMS,YMS_000068,YMS_000068,1.0,NaN,NaN,1.0,NaN,EUR,NaN,...,USA,m2,NIH,DNA from blood,NaN,NaN,NaN,0.0,NaN,0
156939,YMS,YMS_000069,YMS_000069,1.0,NaN,NaN,1.0,NaN,EUR,NaN,...,USA,m2,NIH,DNA from blood,NaN,NaN,NaN,0.0,NaN,1
156941,YMS,YMS_000071,YMS_000071,1.0,NaN,NaN,1.0,NaN,EUR,NaN,...,USA,m2,NIH,DNA from blood,NaN,NaN,NaN,0.0,NaN,1
156945,YMS,YMS_000075,YMS_000075,1.0,NaN,NaN,1.0,NaN,EUR,NaN,...,USA,m2,NIH,DNA from blood,NaN,NaN,NaN,0.0,NaN,1
156948,YMS,YMS_000078,YMS_000078,1.0,NaN,NaN,1.0,NaN,EUR,NaN,...,USA,m3,NIH,Blood (EDTA),NaN,NaN,NaN,0.0,NaN,1


In [35]:
COV = CLINICAL_DF_nba_pruned[['FID','IID','biological_sex_for_qc','genotyping_site']]
COV = COV[['FID','IID','biological_sex_for_qc','genotyping_site']]

In [36]:
PC_ANCESTRIES = []
for ancestry in ANCESTRIES:
    PCs = pd.read_csv(f'{R12_DIR}/raw_genotypes/{ancestry}/{ancestry}_release12_vwb.eigenvec', sep = r'\s+')
    PC_ANCESTRIES.append(PCs)
PC_ANCESTRIES = pd.concat(PC_ANCESTRIES)
PC_ANCESTRIES[PC_ANCESTRIES.duplicated(subset='IID')].shape[0]


0

In [37]:
print(ANC)

FIN


In [38]:
PC_ANCESTRIES.head()

,#FID,IID,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10
0,24HR_000001,24HR_000001,0.001693,0.000683,-0.002432,-0.002203,-0.000061,-0.000458,-0.002965,-0.000632,0.001368,-0.001378
1,24HR_000002,24HR_000002,0.000633,0.005570,0.003405,0.003138,0.000075,0.005095,-0.000211,-0.003239,0.004243,-0.000791
2,24HR_000003,24HR_000003,0.001964,0.001830,-0.000801,-0.000681,0.001047,0.001566,0.002190,0.002200,-0.000910,-0.000624
3,24HR_000004,24HR_000004,-0.005769,0.001013,-0.002581,-0.000608,-0.001305,0.002956,0.000364,0.000199,-0.000450,0.000911
4,24HR_000005,24HR_000005,0.002357,-0.002457,-0.001501,-0.000811,-0.003234,0.002135,-0.001189,0.004278,0.001888,-0.000872


In [39]:
print(ANC)

FIN


In [40]:
COV_all_wPCs = COV.merge(
    PC_ANCESTRIES.drop(columns=['#FID'])
)
COV_all_wPCs.head()

,FID,IID,biological_sex_for_qc,genotyping_site,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10
0,24HR_000001,24HR_000001,Female,Psomagen,0.001693,0.000683,-0.002432,-0.002203,-0.000061,-0.000458,-0.002965,-0.000632,0.001368,-0.001378
1,24HR_000002,24HR_000002,Male,Psomagen,0.000633,0.005570,0.003405,0.003138,0.000075,0.005095,-0.000211,-0.003239,0.004243,-0.000791
2,24HR_000003,24HR_000003,Male,Psomagen,0.001964,0.001830,-0.000801,-0.000681,0.001047,0.001566,0.002190,0.002200,-0.000910,-0.000624
3,24HR_000004,24HR_000004,Male,Psomagen,-0.005769,0.001013,-0.002581,-0.000608,-0.001305,0.002956,0.000364,0.000199,-0.000450,0.000911
4,24HR_000005,24HR_000005,Male,Psomagen,0.002357,-0.002457,-0.001501,-0.000811,-0.003234,0.002135,-0.001189,0.004278,0.001888,-0.000872


In [41]:
COV_all_wPCs['biological_sex_for_qc'].unique()

array(['Female', 'Male'], dtype=object)

In [42]:
COV_all_wPCs.to_csv('data/COV_PHENO/COV_ALL_ANC_R12.txt', sep = '\t', index = None, na_rep = 'NA')

# 5. REGENIE Step 1. Ridge regression.

Ok a quick primer on Regenie. REGENIE is a two step process.

1. Step 1: Run a whole-genome regression using unimputed genotyping data. This essentially creates an offset that allows you to include related participants + increase power by residualizing whole-genome effects outside of the chromosome of analysis.
2. Step 2: Actual GWAS.

## 5a. Extract variants and samples that meet the QC criteria for Regenie Step 1

Step 1 has a more stringent QC requirement.
* MAF > 1%
* MAC > 100
* genotyping rate > 90%
* HWE 1e-15
* Autosomes only
* Exclusion of ENCODE blacklist (low complexity regions): https://github.com/Boyle-Lab/Blacklist/tree/master/lists

You can use LD pruning to reduce number of variants. Ideal range is 450k-500k. https://github.com/rgcgithub/regenie/issues/383

**You only need to DO QC STEPS ONCE per ancestry (per release)** If you need to rerun everything because of changes to phenotype, covariates, etc. don't worry about redoing QCs for step 1 file.

In [43]:
# Keep the ancestry selected in the collaborator configuration cell.
print(ANC)

FIN


In [44]:
!mkdir -p data/Regenie_step1_files
!pwd
print(f"Using blacklist: {BLACKLIST_BED}")
!ls


/home/jupyter/workspace/ws_files/MAMA_2/jf_release12/test
Using blacklist: /home/jupyter/workspace/ws_files/MAMA_2/FirstTryApril26/data/hg38-blacklist.v2.bed
chr10_FIN_release12_vwb.pvar  FIN_release12_vwb.pvar  Regenie_GWAS_EUR.ipynb
data			      output


In [45]:
EXTRACT_STEP1_VARS_CMD = f'mkdir -p data/Regenie_step1_files/{ANC}/; \
{PLINK2} --pfile "{R12_DIR}/raw_genotypes/{ANC}/{ANC}_release12_vwb" \
  --maf 0.01 --mac 100 --geno 0.1 --hwe 1e-15 \
  --autosome \
  --indep-pairwise 1000 100 0.8 \
  --exclude range "{BLACKLIST_BED}" \
  --write-snplist --write-samples \
  --out data/Regenie_step1_files/{ANC}/regenie_step1_var_inclusion'
!$EXTRACT_STEP1_VARS_CMD

PLINK v2.0.0-a.6.9LM 64-bit Intel (29 Jan 2025)    cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to data/Regenie_step1_files/FIN/regenie_step1_var_inclusion.log.
Options in effect:
  --autosome
  --exclude range /home/jupyter/workspace/ws_files/MAMA_2/FirstTryApril26/data/hg38-blacklist.v2.bed
  --geno 0.1
  --hwe 1e-15
  --indep-pairwise 1000 100 0.8
  --mac 100
  --maf 0.01
  --out data/Regenie_step1_files/FIN/regenie_step1_var_inclusion
  --pfile /home/jupyter/workspace/gp2_tier2_eu_release12/raw_genotypes/FIN/FIN_release12_vwb
  --write-samples
  --write-snplist

Start time: Thu Sep  3 14:25:48 2026
52195 MiB RAM detected, ~18200 available; reserving 18136 MiB for main
workspace.
Using up to 8 compute threads.
183 samples (103 females, 80 males; 183 founders) loaded from
/home/jupyter/workspace/gp2_tier2_eu_release12/raw_genotypes/FIN/FIN_release12_vwb.psam.
1824691 out of 1901389 variants loaded from
/home/jupyte

In [46]:
# !plink2 --pfile "{R11_DIR}/raw_genotypes//AMR_release11_vwb" \
#   --maf 0.01 --mac 100 --geno 0.1 --hwe 1e-15 0 \
#   --indep-pairwise 1000 100 0.8 \
#   --exclude range "/hg38-blacklist.v2.bed" \
#   --write-snplist --write-samples \
#   --out scratch/qc_pass_1

~500k variants post LD pruning, acceptable. We will proceed.

In [47]:
!mkdir -p data/R12_regenie_ready_genotypes

In [48]:
MAKE_FILE_CMD = f"{PLINK2} --pfile {R12_DIR}/raw_genotypes/{ANC}/{ANC}_release12_vwb \
--extract data/Regenie_step1_files/{ANC}/regenie_step1_var_inclusion.prune.in \
--keep data/Regenie_step1_files/{ANC}/regenie_step1_var_inclusion.id \
--make-pgen psam-cols=+fid --out data/R12_regenie_ready_genotypes/{ANC}_release12_vwb"
!$MAKE_FILE_CMD

PLINK v2.0.0-a.6.9LM 64-bit Intel (29 Jan 2025)    cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to data/R12_regenie_ready_genotypes/FIN_release12_vwb.log.
Options in effect:
  --extract data/Regenie_step1_files/FIN/regenie_step1_var_inclusion.prune.in
  --keep data/Regenie_step1_files/FIN/regenie_step1_var_inclusion.id
  --make-pgen psam-cols=+fid
  --out data/R12_regenie_ready_genotypes/FIN_release12_vwb
  --pfile /home/jupyter/workspace/gp2_tier2_eu_release12/raw_genotypes/FIN/FIN_release12_vwb

Start time: Thu Sep  3 14:26:27 2026
52195 MiB RAM detected, ~18229 available; reserving 18165 MiB for main
workspace.
Using up to 8 compute threads.
183 samples (103 females, 80 males; 183 founders) loaded from
/home/jupyter/workspace/gp2_tier2_eu_release12/raw_genotypes/FIN/FIN_release12_vwb.psam.
1901389 variants loaded from
/home/jupyter/workspace/gp2_tier2_eu_release12/raw_genotypes/FIN/FIN_release12_vwb.pvar.
1 binary

## 5b. Actually run step 1

After testing, it looks like Regenie generates the whole genome model off of all participants, even if some are missing in the phenotype.

So when you are preparing to run GWAS(es) or gene burden that only uses a subset of the participants, MAKE SURE TO ADD `--keep` command with the phenotype file so that it only runs the whole genome model off of the subset of the participants.

**LOOCV vs K-fold CV**

Cross-validation is necessary to prevent proximal contamination. There are two ways to do it: leave-one-out and k-fold (typically k=5).

While the performance of the two are similar, LOOCV is a bit faster with binary traits while k-fold is faster for quantitative. Use LOOCV (by adding `--loocv` option) when number of samples are < 5000 or using binary traits.

**Note on performance**

`--lowmem` has a small impact on performance, but necessary RAM can get pretty high depending on the number of phenotypes and covariates (see https://dnanexus.gitbook.io/uk-biobank-rap/science-corner/gwas-ex#benchmarking; 10 quantitative phenotypes with 3 covariates can consume ~100Gb of RAM in UKB, and binary may use more RAM). So only leave that option out if you are testing low number of phenotypes.

In [49]:
MKDIR_STEP1OUTPUT = f"mkdir -p data/Regenie_step1_output/"
!$MKDIR_STEP1OUTPUT
STEP1_CMD = f'{REGENIE} --step 1 \
  --pgen data/R12_regenie_ready_genotypes/{ANC}_release12_vwb \
  --extract data/Regenie_step1_files/{ANC}/regenie_step1_var_inclusion.prune.in \
  --keep data/COV_PHENO/GP2_R12_ALLANC_PD_PHENO.regenie.txt \
  --phenoFile data/COV_PHENO/GP2_R12_ALLANC_PD_PHENO.regenie.txt \
  --use-relative-path \
  --covarFile data/COV_PHENO/COV_ALL_ANC_R12.txt \
  --covarColList PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10 \
  --catCovarList biological_sex_for_qc \
  --bt \
  --threads {THREADS} \
  --gz \
  --bsize 1000 \
  --lowmem \
  --loocv \
  --out data/Regenie_step1_output/GP2_R12_{ANC}_imputed'
!$STEP1_CMD

Start time: Thu Sep  3 14:26:48 2026

              |=============================|
              |      REGENIE v4.1.2.gz      |
              |=============================|

Copyright (c) 2020-2024 Joelle Mbatchou, Andrey Ziyatdinov and Jonathan Marchini.
Distributed under the MIT License.
Compiled with Boost Iostream library.
Using Intel MKL with Eigen.

Log of output saved in file : data/Regenie_step1_output/GP2_R12_FIN_imputed.log

Options in effect:
  --step 1 \
  --pgen data/R12_regenie_ready_genotypes/FIN_release12_vwb \
  --extract data/Regenie_step1_files/FIN/regenie_step1_var_inclusion.prune.in \
  --keep data/COV_PHENO/GP2_R12_ALLANC_PD_PHENO.regenie.txt \
  --phenoFile data/COV_PHENO/GP2_R12_ALLANC_PD_PHENO.regenie.txt \
  --use-relative-path \
  --covarFile data/COV_PHENO/COV_ALL_ANC_R12.txt \
  --covarColList PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10 \
  --catCovarList biological_sex_for_qc \
  --bt \
  --threads 8 \
  --gz \
  --bsize 1000 \
  --lowmem \
  --loocv \
  -

Step 1 took only 445.611s seconds. Expect slower as sample size gets larger (UKB can take hours).

# 6. Step 2 - actual GWAS

REGENIE doesn't have MAF filter, only MAC.

For test, we will only look at INFO > 0.6

In [50]:
MAC = np.floor((899+628)*0.01)
MAC

15.0

In [51]:
### ASK ABOUT WHY THIS MAC

In [53]:
print(ANC)

FIN


In [54]:
CMD1 = f'mkdir -p data/symlink_R12_{ANC}; \
ln -s $(realpath {R12_DIR}/imputed_genotypes/{ANC}/chr*_{ANC}_release12_vwb.'
CMD2 = '{pvar,pgen}) '
CMD3 = f'data/symlink_R12_{ANC}/'
SYMLINK_CMD = CMD1+CMD2+CMD3
!$SYMLINK_CMD

# CMD1 = f'mkdir -p data/symlink_R11_{ANC}; \
# ln -s $(realpath {R11_DIR}/imputed_genotypes/{ANC}/chr*_{ANC}_release11_vwb.'
# CMD2 = '{pvar,pgen}) '
# CMD3 = f'data/symlink_R11_{ANC}/'
# SYMLINK_CMD = CMD1+CMD2+CMD3
# !bash ls "$SYMLINK_CMD"

# SYMLINK_CMD = (
#     f"mkdir -p data/symlink_R11_{ANC}; "
#     f"ln -s $(realpath {R11_DIR}/imputed_genotypes/{ANC}/chr*_{ANC}_release11_vwb.{{pvar,pgen}}) "
#     f"data/symlink_R11_{ANC}/"
# )

# print(SYMLINK_CMD)
# !bash -lc "$SYMLINK_CMD"

In [56]:
for CHRNUM in CHROMOSOMES:
    CMD = f'{PLINK2} --pfile {R12_DIR}/imputed_genotypes/{ANC}/chr{CHRNUM}_{ANC}_release12_vwb \
    --write-snplist \
    --make-just-psam cols=+fid --out data/symlink_R12_{ANC}/chr{CHRNUM}_{ANC}_release12_vwb'
    !$CMD

PLINK v2.0.0-a.6.9LM 64-bit Intel (29 Jan 2025)    cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to data/symlink_R12_FIN/chr1_FIN_release12_vwb.log.
Options in effect:
  --make-just-psam cols=+fid
  --out data/symlink_R12_FIN/chr1_FIN_release12_vwb
  --pfile /home/jupyter/workspace/gp2_tier2_eu_release12/imputed_genotypes/FIN/chr1_FIN_release12_vwb
  --write-snplist

Start time: Thu Sep  3 14:33:13 2026
52195 MiB RAM detected, ~18181 available; reserving 18117 MiB for main
workspace.
Using up to 8 compute threads.
183 samples (103 females, 80 males; 183 founders) loaded from
/home/jupyter/workspace/gp2_tier2_eu_release12/imputed_genotypes/FIN/chr1_FIN_release12_vwb.psam.
1079262 variants loaded from
/home/jupyter/workspace/gp2_tier2_eu_release12/imputed_genotypes/FIN/chr1_FIN_release12_vwb.pvar.
1 binary phenotype loaded (126 cases, 20 controls).
--write-snplist: Variant IDs written to
data/symlink_R12_FIN/chr1_FIN_re

In [61]:
MKDIR_CMD = f'mkdir -p output/Regenie_GWAS/{ANC}_R12_imputed/'
!$MKDIR_CMD


In [62]:
### For this step use 16 CPUs

In [63]:
for CHRNUM in STEP2_CHROMOSOMES:
    CMD = f'{REGENIE} --step 2 \
      --pred data/Regenie_step1_output/GP2_R12_{ANC}_imputed_pred.list \
      --pgen "data/symlink_R12_{ANC}/chr{CHRNUM}_{ANC}_release12_vwb" \
      --keep data/COV_PHENO/GP2_R12_ALLANC_PD_PHENO.regenie.txt \
      --phenoFile data/COV_PHENO/GP2_R12_ALLANC_PD_PHENO.regenie.txt \
      --use-relative-path \
      --covarFile data/COV_PHENO/COV_ALL_ANC_R12.txt \
      --covarColList PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10 \
      --catCovarList biological_sex_for_qc \
      --minMAC {MAC} \
      --minINFO 0.6 \
      --bt \
      --af-cc \
      --threads {THREADS} \
      --firth --firth-se --approx \
      --gz \
      --bsize 200 \
      --out output/Regenie_GWAS/{ANC}_R12_imputed/{CHRNUM}'
    !$CMD
    

SYMLINK_CMD = (
    f"mkdir -p data/symlink_R12_{ANC}; "
    f"ln -s $(realpath {R12_DIR}/imputed_genotypes/{ANC}/chr*_{ANC}_release12_vwb.{{pvar,pgen}}) "
    f"data/symlink_R12_{ANC}/"
)

print(SYMLINK_CMD)
!bash -lc "$SYMLINK_CMD"



Start time: Thu Sep  3 14:34:29 2026

              |=============================|
              |      REGENIE v4.1.2.gz      |
              |=============================|

Copyright (c) 2020-2024 Joelle Mbatchou, Andrey Ziyatdinov and Jonathan Marchini.
Distributed under the MIT License.
Compiled with Boost Iostream library.
Using Intel MKL with Eigen.

Log of output saved in file : output/Regenie_GWAS/FIN_R12_imputed/1.log

Options in effect:
  --step 2 \
  --pred data/Regenie_step1_output/GP2_R12_FIN_imputed_pred.list \
  --pgen data/symlink_R12_FIN/chr1_FIN_release12_vwb \
  --keep data/COV_PHENO/GP2_R12_ALLANC_PD_PHENO.regenie.txt \
  --phenoFile data/COV_PHENO/GP2_R12_ALLANC_PD_PHENO.regenie.txt \
  --use-relative-path \
  --covarFile data/COV_PHENO/COV_ALL_ANC_R12.txt \
  --covarColList PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10 \
  --catCovarList biological_sex_for_qc \
  --minMAC 15.0 \
  --minINFO 0.6 \
  --bt \
  --af-cc \
  --threads 8 \
  --firth \
  --firth-se \
  --app

In [64]:
def fix_regenie_output(df):
    df['A1FREQ_CONTROLS'] = (
        (df['A1FREQ']*df['N']-df['A1FREQ_CASES']*df['N_CASES'])
        /df['N_CONTROLS'])
    df['P'] = np.power(10,-df['LOG10P'])
    return(df)

def read_regenie_output(path, gene_level=False):
    if gene_level:
        DF = pd.read_csv(path, sep = ' ', skiprows=1)
    else:
        DF = pd.read_csv(path, sep = ' ')
    DF = fix_regenie_output(DF)
    return(DF)

Only issue with REGENIE right now - their "A1FREQ_CONTROLS" column is bugged. Older versions don't have this bug, but they're also slower. We can fix this easily by reading the outputs with the functions above.

In [65]:
GWAS_RES = []
for CHRNUM in CHROMOSOMES:
    TEMP = read_regenie_output(
        f"output/Regenie_GWAS/{ANC}_R12_imputed/{CHRNUM}_PD.regenie.gz"
    )
    GWAS_RES.append(TEMP)

In [66]:
GWAS_RES = pd.concat(GWAS_RES)
GWAS_RES.to_csv(
    f"output/Regenie_GWAS/{ANC}_R12_imputed/FULL_{ANC}_GWAS_PD.regenie.gz",
    sep = '\t',
    index = None,
    na_rep = 'NA'
)
GWAS_RES.sort_values(by='P')

,CHROM,GENPOS,ID,ALLELE0,ALLELE1,A1FREQ,A1FREQ_CASES,A1FREQ_CONTROLS,INFO,N,N_CASES,N_CONTROLS,TEST,BETA,SE,CHISQ,LOG10P,EXTRA,P
242910,8,108017070,chr8:108017070:G:C,G,C,0.186900,0.142204,0.442040,0.862259,161,137,24,ADD,-1.901000e+00,0.493529,1.483680e+01,3.930980e+00,NaN,0.000117
61953,22,45880448,chr22:45880448:T:G,T,G,0.388108,0.340277,0.661143,1.102930,161,137,24,ADD,-1.301440e+00,0.366170,1.263230e+01,3.421200e+00,NaN,0.000379
242894,8,108008602,chr8:108008602:A:C,A,C,0.288820,0.244526,0.541665,0.986993,161,137,24,ADD,-1.325880e+00,0.379152,1.222860e+01,3.327330e+00,NaN,0.000471
242895,8,108008782,chr8:108008782:A:C,A,C,0.287426,0.244037,0.535105,0.969582,161,137,24,ADD,-1.338090e+00,0.383498,1.217440e+01,3.314700e+00,NaN,0.000485
242904,8,108014750,chr8:108014750:C:T,C,T,0.287835,0.244084,0.537580,0.973981,161,137,24,ADD,-1.333610e+00,0.382785,1.213810e+01,3.306260e+00,NaN,0.000494
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259217,5,121514529,chr5:121514529:A:C,A,C,0.322568,0.324321,0.312561,0.943968,161,137,24,ADD,-9.060030e-08,0.437940,4.279860e-14,7.168680e-08,NaN,1.000000
162152,14,92912806,chr14:92912806:A:C,A,C,0.633560,0.621983,0.699645,0.911372,161,137,24,ADD,-8.319820e-08,0.513518,2.624920e-14,5.614130e-08,NaN,1.000000
179356,12,83987692,chr12:83987692:C:T,C,T,0.181869,0.180988,0.186898,1.015630,161,137,24,ADD,-3.698230e-08,0.501406,5.440120e-15,2.555810e-08,NaN,1.000000
106691,12,48061472,chr12:48061472:G:A,G,A,0.513836,0.516479,0.498749,0.993149,161,137,24,ADD,2.573800e-08,0.397146,4.200010e-15,2.245690e-08,NaN,1.000000


Novel hit in CAS? 🤔